<a href="https://colab.research.google.com/github/Viswas-Vinayakumar/AI-search/blob/main/Keyword_vector_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!python -m spacy download de_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 MB 20.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# !python -m spacy download de_core_news_md -q      <- run this once in Colab first
import math, re
import numpy as np
import spacy

# ===== 1. DATA: 4 coaches, 3 searches, and the correct answer for each =====
COACHES = {
    "A": "PRINCE2 Zertifizierung und klassisches Projektmanagement",
    "B": "Agile Projekte mit Scrum leiten",
    "C": "Konflikte im Team lösen und Mediation",
    "D": "Stress reduzieren und Burnout vorbeugen",
}
GOLDEN = {
    "PRINCE2 Kurs":               "A",   # exact technical name
    "Streit unter Kollegen":      "C",   # same meaning, no shared words
    "Erschöpfung bei der Arbeit": "D",   # same meaning, no shared words
}

# ===== 2. KEYWORD SEARCH (BM25) =====
STOP = {"und", "im", "mit", "bei", "der", "unter"}
def tokenize(text):
    return [w for w in re.findall(r"\w+", text.lower()) if w not in STOP]

docs = {cid: tokenize(t) for cid, t in COACHES.items()}
N = len(docs)
avg_len = sum(len(d) for d in docs.values()) / N
k1, b = 1.5, 0.75

def idf(word):
    n = sum(word in d for d in docs.values())
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

def bm25(query, doc):
    score = 0.0
    for w in tokenize(query):
        tf = doc.count(w)
        if tf:
            score += idf(w) * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(doc) / avg_len))
    return score

def keyword_rank(query):
    scored = sorted(((cid, bm25(query, d)) for cid, d in docs.items()), key=lambda x: -x[1])
    return [(cid, s) for cid, s in scored if s > 0]          # score 0 = no shared word = not found

# ===== 3. VECTOR SEARCH (semantic) =====
nlp = spacy.load("de_core_news_md")
def embed(text):
    vecs = [t.vector for t in nlp(text) if t.is_alpha and not t.is_stop and t.has_vector]
    v = np.mean(vecs, axis=0)                                 # average the word vectors
    return v / np.linalg.norm(v)                              # length 1

coach_vecs = {cid: embed(t) for cid, t in COACHES.items()}
def vector_rank(query):
    q = embed(query)
    return sorted(((cid, float(v @ q)) for cid, v in coach_vecs.items()), key=lambda x: -x[1])

# ===== 4. HYBRID (RRF) =====
def hybrid_rank(query, k=60):
    fused = {}
    for ranking in (keyword_rank(query), vector_rank(query)):
        for rank, (cid, _) in enumerate(ranking, start=1):
            fused[cid] = fused.get(cid, 0.0) + 1 / (k + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

METHODS = {"keyword": keyword_rank, "vector": vector_rank, "hybrid": hybrid_rank}

# ===== 5. RESULTS PER METHOD =====
def position(ranked, correct):
    ids = [cid for cid, _ in ranked]
    return ids.index(correct) + 1 if correct in ids else None

for name, fn in METHODS.items():
    print(f"\n===== {name.upper()} =====")
    for q, correct in GOLDEN.items():
        ranked = fn(q)
        pos = position(ranked, correct)
        verdict = "nothing found" if not ranked else ("correct at #1" if pos == 1 else f"wrong: correct coach at #{pos}")
        shown = [(cid, round(s, 4)) for cid, s in ranked]
        print(f"{q:<28} -> {shown}   [{verdict}]")

# ===== 6. RRF CALCULATION, STEP BY STEP, FOR ONE QUERY =====
q = "PRINCE2 Kurs"
kw = {cid: r for r, (cid, _) in enumerate(keyword_rank(q), start=1)}
vc = {cid: r for r, (cid, _) in enumerate(vector_rank(q), start=1)}
print(f"\n===== RRF STEP BY STEP: '{q}' =====")
print("coach | keyword rank | vector rank | calculation          | RRF")
for cid, score in hybrid_rank(q):
    kpart = f"1/{60 + kw[cid]}" if cid in kw else "0"
    vpart = f"1/{60 + vc[cid]}" if cid in vc else "0"
    print(f"{cid:<5} | {str(kw.get(cid, '-')):<12} | {str(vc.get(cid, '-')):<11} | {kpart + ' + ' + vpart:<20} | {score:.4f}")

# ===== 7. SCOREBOARD: position of the correct coach + MRR =====
print("\n===== SCOREBOARD =====")
print(f"{'query':<28} | {'keyword':<9} | {'vector':<9} | {'hybrid':<9}")
rr = {name: [] for name in METHODS}
for q, correct in GOLDEN.items():
    cells = []
    for name, fn in METHODS.items():
        pos = position(fn(q), correct)
        rr[name].append(1 / pos if pos else 0.0)
        cells.append("not found" if pos is None else f"#{pos}")
    print(f"{q:<28} | {cells[0]:<9} | {cells[1]:<9} | {cells[2]:<9}")
for name in METHODS:
    parts = " + ".join(f"{x:.2f}" for x in rr[name])
    print(f"MRR {name:<8} = ({parts}) / {len(GOLDEN)} = {np.mean(rr[name]):.2f}")


===== KEYWORD =====
PRINCE2 Kurs                 -> [('A', 1.204)]   [correct at #1]
Streit unter Kollegen        -> []   [nothing found]
Erschöpfung bei der Arbeit   -> []   [nothing found]

===== VECTOR =====
PRINCE2 Kurs                 -> [('C', 0.2626), ('B', 0.2564), ('A', 0.2244), ('D', 0.1455)]   [wrong: correct coach at #3]
Streit unter Kollegen        -> [('C', 0.528), ('D', 0.2935), ('B', 0.2728), ('A', 0.0927)]   [correct at #1]
Erschöpfung bei der Arbeit   -> [('B', 0.4717), ('C', 0.4698), ('D', 0.457), ('A', 0.4411)]   [wrong: correct coach at #3]

===== HYBRID =====
PRINCE2 Kurs                 -> [('A', 0.0323), ('C', 0.0164), ('B', 0.0161), ('D', 0.0156)]   [correct at #1]
Streit unter Kollegen        -> [('C', 0.0164), ('D', 0.0161), ('B', 0.0159), ('A', 0.0156)]   [correct at #1]
Erschöpfung bei der Arbeit   -> [('B', 0.0164), ('C', 0.0161), ('D', 0.0159), ('A', 0.0156)]   [wrong: correct coach at #3]

===== RRF STEP BY STEP: 'PRINCE2 Kurs' =====
coach | keyword ran